## Verificación real del cruce — 7 de octubre de 2026

El registro de Prefect mostró `KeyboardInterrupt` durante B, después de que A terminara.
No se pudo atribuir la interrupción a un fallo automático de workers. Se probaron tres
arquitecturas fuera del notebook, sobre los intermedios del Silver existente:

| Arquitectura del cruce | Tiempo total de prueba | Resultado |
| --- | --- | --- |
| P2P, 1 worker × 4 GiB | 15.46 s | Completado |
| Tareas, 2 workers × 3 GiB | 9.15 s | Completado |
| Tareas, 1 worker × 4 GiB | 13.49 s | Completado |

Las tres salidas tienen **626.821 filas e IDs distintos**, **525.835 coincidencias
contractuales** y **539.919 objetivos positivos**. La comparación bidireccional
`EXCEPT ALL` de todas las columnas produjo **0 filas diferentes**. Una referencia
SQL independiente del JOIN y objetivo también produjo **0 diferencias**.

Se eligió shuffle `tasks`, preparación con 2 workers × 2 GiB y cruce con 1 worker × 4 GiB.
El cruce con dos workers fue más rápido en esta medición, pero exigió 6 GiB en vez de
4 GiB; se priorizó el presupuesto conservador. Una sola medición no prueba optimalidad.

Después se ejecutó `build_gold(audit=..., publish=False)` con ambas etapas y memoria
automática. Terminó en **32.17 segundos**: preparación
**17.27 s**, cruce/auditoría
**11.96 s**; el total incluye arranque/cierre.
Se comprobó mediante SHA-256 que los Parquet Gold publicados permanecieron intactos.
No se descargaron datos ni se cargó PostgreSQL. Esto valida Gold aislado; no es una
nueva ejecución completa de Prefect ni garantiza que una interrupción externa no se repita.

**Para ejecutar en Jupyter:** reiniciar el kernel y ejecutar las celdas iniciales del
notebook 05 para cargar el módulo nuevo. Ahora Gold imprime avances de lectura,
preparación, JOIN, escritura y auditoría. Los mensajes de shuffle marcados como WARNING
no significan por sí solos que el flujo haya fallado.

Verificación opcional sin publicar, si se necesita repetir la comprobación:

```python
from secop_pipeline.dask_jobs import build_gold
evidencia = {}
build_gold(audit=evidencia, publish=False)
```

`publish=False` construye temporalmente ambas etapas, valida conteos y descarta la
salida. Devuelve `None`. La ejecución normal usa `publish=True` y devuelve la ruta Gold.


## Ajuste de memoria tras el fallo de ejecución

El fallo registrado ocurrió antes de crear workers: la reserva fija de 8 GiB superó
el 70% de los 11 GiB disponibles. Se reemplazó por un presupuesto automático de hasta
4 GiB por etapa, limitado al 50% de RAM disponible. Preparación usa 2 workers; cruce 1.
Con memoria suficiente son 2 GiB y 4 GiB por worker respectivamente. Los límites pueden
bajar y se registran en el informe. Para modo automático, no pasar `memory_limit` ni
`--dask-memory` y dejar vacías DASK_PREP_MEMORY/DASK_JOIN_MEMORY.

Reiniciar el kernel antes de ejecutar para cargar el código nuevo. Las salidas guardadas
de ejecuciones anteriores no prueban que este ajuste se haya ejecutado correctamente.
No hace falta volver a ejecutar los notebooks 01–04: Prefect reutiliza Silver válido.


# 03 — Dask en dos etapas complementarias del ETL

Gold se construye una sola vez. Configuraciones iniciales para el equipo observado:

| Etapa | Trabajo útil | Workers | Memoria por worker | Total solicitado |
| --- | --- | --- | --- | --- |
| A: preparación | Filtrar procesos, deduplicar IDs y claves contractuales, escribir intermedios | 2 | Hasta 2 GiB | Hasta 4 GiB |
| B: cruce Gold | Leer intermedios de A, LEFT JOIN, variables derivadas y escritura Gold | 1 | Hasta 4 GiB | Hasta 4 GiB |

Equipo observado: 31,43 GiB RAM total, 14,79 GiB disponibles en la inspección, 16 núcleos físicos
y 22 lógicos. Silver: 9.070.082 procesos y 5.642.709 contratos en 234 archivos, unos 686 MiB
comprimidos. El tamaño descomprimido puede ser muy superior.

A aprovecha lectura y limpieza de particiones con dos procesos; B da más margen por
worker al cruce y sus redistribuciones. Un hilo por worker limita operaciones pesadas
simultáneas. Los clústeres son secuenciales y sus memorias no se suman. Estos valores son
una elección inicial razonada, NO un óptimo medido. La verificación aislada de Gold y sus límites se documenta al principio del notebook.

Al ejecutar las celdas se reconstruye Gold local. No se descarga la API ni se carga PostgreSQL.
Para registrar todo el flujo en el informe 07, ejecutar el notebook 05.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)


In [ ]:
from secop_pipeline.dask_jobs import build_gold
import pandas as pd

evidencia_gold = {}
gold_path = build_gold(audit=evidencia_gold)
display(pd.DataFrame([
    {k: v for k, v in etapa.items() if k != "salidas"}
    for etapa in evidencia_gold["profiles"]
]))


## Cómo interpretar la evidencia

La salida de A es la entrada de B. Comparar entradas, salidas, workers observados,
límites efectivos, duración y conservación de filas del JOIN. Los tiempos incluyen
las mediciones de auditoría; el total añade apertura y cierre del clúster.
Como las etapas hacen trabajos distintos, NO dividir tiempos para afirmar que una
configuración es más rápida. Los límites no representan el pico real de memoria.
En modo automático se reduce la memoria antes de cada etapa según disponibilidad, y se registra el ajuste. Solo se detiene si no alcanza el mínimo de 512 MiB por worker.
Los intermedios temporales se eliminan al finalizar; Gold anterior se conserva hasta
que B termine y la conservación de filas se compruebe.

In [ ]:
display(pd.Series(evidencia_gold["counts"], name="conteos_ETL"))
display(pd.Series(evidencia_gold["hardware"], name="recursos_observados"))


In [ ]:
import dask.dataframe as dd
df = dd.read_parquet(str(gold_path / "*.parquet"))
df[["objetivo", "precio_base", "dias_para_cerrar"]].astype("float64").describe().compute()
